In [14]:
!git clone https://github.com/AMAAI-Lab/text2midi
!cd text2midi
!pip install -r text2midi/requirements.txt

Cloning into 'text2midi'...
remote: Enumerating objects: 169, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 169 (delta 38), reused 10 (delta 6), pack-reused 100 (from 1)
Receiving objects: 100% (169/169), 14.29 MiB | 5.35 MiB/s, done.
Resolving deltas: 100% (78/78), done.
Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/main.py", line 78, in main
    command = create_command(cmd_name, isolated=("--isolated" in cmd_args))
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/__init__.py", line 114, in create_command
    module = importlib.import_module(module_path)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/importlib/__init__.py", line 126, in import_m

In [15]:
!cd text2midi

In [16]:
!pip install --upgrade torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118


In [17]:
!pip install music21 librosa pretty_midi laion-clap scipy numpy
!pip install numpy==1.26.4

In [18]:
import os
import pickle
import torch
import torch.nn as nn
from transformers import T5Tokenizer
from huggingface_hub import hf_hub_download
import subprocess
import sys
#import pretty_midi
#import music21
import soundfile as sf
import numpy as np

In [19]:
os.chdir('text2midi')

In [24]:
# Cell 1: Import Libraries and Setup
import os
import pickle
import torch
import torch.nn as nn
from transformers import T5Tokenizer
#from model.transformer_model import Transformer # Now, the import should work
from huggingface_hub import hf_hub_download
import music21
import numpy as np
from scipy.spatial.distance import cosine
import librosa;
import pretty_midi
from typing import List, Dict, Tuple
import json
import random
from datasets import load_dataset
import tempfile
import requests

# CLAP imports - you may need to install these
try:
  import laion_clap
except ImportError:
    print("Warning: laion_clap not installed. Install with: pip install laion-clap")
    laion_clap = None

ModuleNotFoundError: No module named 'numpy.char'

In [ ]:
# Cell 2: Download and Load Model
repo_id = "amaai-lab/text2midi"
# Download the model.bin file
model_path = hf_hub_download(repo_id=repo_id, filename="pytorch_model.bin")
# Download the vocab_remi.pkl file
tokenizer_path = hf_hub_download(repo_id=repo_id, filename="vocab_remi.pkl")

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f"Using device: {device}")

In [ ]:
# Cell 3: Initialize Tokenizer and Model
# Load the tokenizer dictionary
with open(tokenizer_path, "rb") as f:
    r_tokenizer = pickle.load(f)

# Get the vocab size
vocab_size = len(r_tokenizer)
print("Vocab size: ", vocab_size)
model = Transformer(vocab_size, 768, 8, 2048, 18, 1024, False, 8, device=device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
print('Model loaded.')

In [ ]:
#Φόρτωση μοντέλου

import pickle
import torch
import torch.nn as nn
import sys
import os

from transformers import T5Tokenizer
from transformers import AutoTokenizer

# Add the cloned repository directory to the system path
if 'text2midi' not in sys.path:
    sys.path.append('text2midi')

from model.transformer_model import Transformer
from huggingface_hub import hf_hub_download


#Εδω κανω unfreeze καποια layers του Encoder, πρεπει να προσθέσω Data Loading, Optimizer and Loss Function και μια training loop

#from transformers import FlanT5ForConditionalGeneration
#model = FlanT5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

# Freeze all parameters first
#for param in model.parameters():
#    param.requires_grad = False

# Unfreeze the last 4 encoder and decoder block layers
#for i in range(8, 12):  # For 12-layer model (base size)
#    for param in model.encoder.block[i].parameters():
#        param.requires_grad = True
#    for param in model.decoder.block[i].parameters():
#        param.requires_grad = True



repo_id = "amaai-lab/text2midi"
# Download the model.bin file
model_path = hf_hub_download(repo_id=repo_id, filename="pytorch_model.bin")
# Download the vocab_remi.pkl file
tokenizer_path = hf_hub_download(repo_id=repo_id, filename="vocab_remi.pkl")

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f"Using device: {device}")

# Load the tokenizer dictionary
with open(tokenizer_path, "rb") as f:
    r_tokenizer = pickle.load(f)

# Get the vocab size
vocab_size = len(r_tokenizer)
print("Vocab size: ", vocab_size)
model = Transformer(vocab_size, 768, 8, 2048, 18, 1024, False, 8, device=device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
#tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base") # flan-t5 tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/gemma-7b-it") #gemma tokenizer
print('Model loaded.')

# Enter the text prompt and tokenize it
src = "A melodic electronic song with ambient elements, featuring piano, acoustic guitar, alto saxophone, string ensemble, and electric bass. Set in G minor with a 4/4 time signature, it moves at a lively Presto tempo. The composition evokes a blend of relaxation and darkness, with hints of happiness and a meditative quality."
print('Generating for prompt: ' + src)

inputs = tokenizer(src, return_tensors='pt', padding=True, truncation=True)
input_ids = nn.utils.rnn.pad_sequence(inputs.input_ids, batch_first=True, padding_value=0)
input_ids = input_ids.to(device)
attention_mask =nn.utils.rnn.pad_sequence(inputs.attention_mask, batch_first=True, padding_value=0)
attention_mask = attention_mask.to(device)

# Generate the midi
output = model.generate(input_ids, attention_mask, max_len=2000,temperature = 1.0)
output_list = output[0].tolist()
generated_midi = r_tokenizer.decode(output_list)
generated_midi.dump_midi("output.mid")

In [ ]:
gemma_model_name = "google/gemma-7b-it"

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
gemma_tokenizer = AutoTokenizer.from_pretrained(gemma_model_name)
gemma_encoder = AutoModel.from_pretrained(
    gemma_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
gemma_encoder.eval()

In [ ]:
class Text2MidiModel(nn.Module):
    def __init__(self, encoder, vocab_size, encoder_hidden_size=3072, decoder_hidden_size=768,
                 nhead=8, num_layers=6, ff_dim=2048, max_len=1024):
        super().__init__()
        self.encoder = encoder
        self.project = nn.Linear(encoder_hidden_size, decoder_hidden_size)

        self.embedding = nn.Embedding(vocab_size, decoder_hidden_size)
        self.positional_encoding = nn.Parameter(torch.zeros(1, max_len, decoder_hidden_size))

        decoder_layer = nn.TransformerDecoderLayer(d_model=decoder_hidden_size, nhead=nhead,
                                                   dim_feedforward=ff_dim, batch_first=True)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.output_layer = nn.Linear(decoder_hidden_size, vocab_size)

    def forward(self, text_input, midi_input):
        with torch.no_grad():
            enc_output = self.encoder(**text_input).last_hidden_state
        enc_proj = self.project(enc_output)

        midi_embed = self.embedding(midi_input) + self.positional_encoding[:, :midi_input.size(1), :]
        dec_out = self.decoder(tgt=midi_embed, memory=enc_proj)

        return self.output_layer(dec_out)

model = Text2MidiModel(
    encoder=gemma_encoder,
    vocab_size=vocab_size,
    encoder_hidden_size=gemma_encoder.config.hidden_size,
    decoder_hidden_size=768
).to(device)

print("✅ Μοντέλο Gemma Text2Midi φορτώθηκε επιτυχώς.")

In [ ]:
def encode_text(prompt: str):
    return gemma_tokenizer(prompt, return_tensors="pt").to(device)

def generate_tokens(prompt: str, max_len=100):
    input_text = encode_text(prompt)
    input_ids = [r_tokenizer['<BOS>']]
    generated = []

    for _ in range(max_len):
        inp = torch.tensor([input_ids], device=device)
        logits = model(input_text, inp)
        next_token = torch.argmax(logits[0, -1]).item()
        if next_token == r_tokenizer.get('<EOS>', -1):
            break
        input_ids.append(next_token)
        generated.append(next_token)
    return generated

In [ ]:
def tokens_to_events(tokens: list, id_to_token: dict):
    return [id_to_token[i] for i in tokens if i in id_to_token]

def remi_events_to_midi(events: list):
    import miditok
    from miditok import REMI
    tokenizer = REMI(params=None)
    tokenizer.vocab = list(r_tokenizer.keys())
    tokenizer.token_type_graph = {
        "Pitch": ["Velocity"], "Velocity": ["Duration"],
        "Duration": ["Time-Shift"], "Time-Shift": ["Pitch"]
    }
    tokenizer._create_token_types_graph()
    tokenizer.token_to_id = r_tokenizer
    tokenizer.id_to_token = {v: k for k, v in r_tokenizer.items()}

    midi_obj = tokenizer.decode([events])[0]
    return midi_obj

In [ ]:
def save_and_play_midi(midi_obj, filename="output.mid"):
    midi_obj.write(filename)
    fs = FluidSynth("FluidR3_GM.sf2")
    fs.midi_to_audio(filename, "output.wav")
    return ipd.Audio("output.wav")

In [ ]:
prompt = "A peaceful piano melody at sunset"
generated_token_ids = generate_tokens(prompt, max_len=60)

id_to_token = {v: k for k, v in r_tokenizer.items()}
remi_events = tokens_to_events(generated_token_ids, id_to_token)

midi = remi_events_to_midi(remi_events)
audio = save_and_play_midi(midi)
audio

In [21]:
!pip install numpy==1.26.4